# 📘 Regular Expressions for Data Engineering
## Databricks Notebook — Pattern Matching & Text Processing

**What you'll learn:**
- Regex fundamentals: character classes, quantifiers, groups
- Python re module: match, search, findall, sub, compile
- Data validation: emails, phones, dates, IPs
- Log parsing: Apache, application logs, stack traces
- PySpark regex: regexp_extract, regexp_replace, rlike
- PII detection and masking

**Key rules:**
- Always use raw strings: `r"\d{3}-\d{4}"`
- Use `re.compile()` for patterns used more than once
- Never use regex for HTML/XML parsing

**Prerequisites:** Notebooks 01-15 (techmart_dw)

---

---
# 📗 Section 1: Why Regex for Data Engineering

**Use cases:**
1. **Log parsing:** Extract timestamps, levels, messages from unstructured logs
2. **Validation:** Check email, phone, date formats before ingestion
3. **Cleaning:** Remove special characters, normalize formats
4. **Extraction:** Pull structured data from text fields
5. **PII detection:** Find and mask sensitive data (SSN, credit cards)

---
# 📗 Section 2: Regex Fundamentals

| Pattern | Matches | Example |
|---|---|---|
| `\d` | Any digit | `\d{3}` → "123" |
| `\w` | Word char (letter/digit/_) | `\w+` → "hello_123" |
| `\s` | Whitespace | `\s+` → "  " |
| `.` | Any char (except newline) | `a.c` → "abc", "a1c" |
| `[abc]` | Any of a, b, c | `[aeiou]` → vowels |
| `[^abc]` | NOT a, b, c | `[^0-9]` → non-digits |
| `*` | 0 or more | `a*` → "", "a", "aaa" |
| `+` | 1 or more | `a+` → "a", "aaa" |
| `?` | 0 or 1 | `colou?r` → "color", "colour" |
| `{n,m}` | n to m times | `\d{2,4}` → "12", "1234" |
| `^` | Start of string | `^Hello` |
| `$` | End of string | `world$` |
| `\b` | Word boundary | `\bcat\b` → "cat" not "category" |
| `(...)` | Capture group | `(\d+)-(\d+)` |
| `(?P<name>...)` | Named group | `(?P<year>\d{4})` |

---
# 📗 Section 3: Python re Module

In [0]:
import re

# re.search() — find first match anywhere in string
text = "Order #12345 placed on 2024-06-15 for $299.99"

# Find order number
match = re.search(r"#(\d+)", text)
if match:
    print(f"Order number: {match.group(1)}")

# Find date
date_match = re.search(r"(\d{4}-\d{2}-\d{2})", text)
if date_match:
    print(f"Date: {date_match.group(1)}")

# Find amount
amount_match = re.search(r"\$(\d+\.\d{2})", text)
if amount_match:
    print(f"Amount: ${amount_match.group(1)}")

In [0]:
# re.findall() — find ALL matches
log_text = """
2024-06-15 10:30:01 ERROR auth-service Connection timeout
2024-06-15 10:30:05 INFO order-service Order processed
2024-06-15 10:30:10 ERROR payment-service Gateway error
2024-06-15 10:30:15 WARN inventory-service Low stock
"""

# Find all timestamps
timestamps = re.findall(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}", log_text)
print(f"Timestamps: {timestamps}")

# Find all ERROR lines
errors = re.findall(r".*ERROR.*", log_text)
print(f"Error lines: {errors}")

# Named groups: parse each log line
pattern = r"(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) (?P<level>\w+) (?P<service>[\w-]+) (?P<message>.+)"
for match in re.finditer(pattern, log_text):
    if match.group("level") == "ERROR":
        print(f"  {match.group('service')}: {match.group('message')}")

In [0]:
# re.sub() — replacement
# Mask credit card numbers
text = "Payment with card 4532-1234-5678-9012 approved"
masked = re.sub(r"\d{4}-\d{4}-\d{4}-(\d{4})", r"****-****-****-\1", text)
print(f"Original: {text}")
print(f"Masked:   {masked}")

# Normalize phone numbers
phones = ["(555) 123-4567", "555.123.4567", "555-123-4567", "+1 555 123 4567"]
for phone in phones:
    normalized = re.sub(r"[^\d]", "", phone)[-10:]  # Keep last 10 digits
    formatted = f"({normalized[:3]}) {normalized[3:6]}-{normalized[6:]}"
    print(f"  {phone:<20} → {formatted}")

In [0]:
# re.compile() — pre-compile for performance
import re

# Compile once, use many times
EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
PHONE_PATTERN = re.compile(r"\+?1?[-. ]?\(?\d{3}\)?[-. ]?\d{3}[-. ]?\d{4}")

test_strings = [
    "Contact john@example.com or call (555) 123-4567",
    "Email: jane.doe@company.co.uk",
    "No contact info here",
]

for s in test_strings:
    emails = EMAIL_PATTERN.findall(s)
    phones = PHONE_PATTERN.findall(s)
    print(f"  '{s[:40]}...' → emails={emails}, phones={phones}")

---
# 📗 Section 4: Data Validation with Regex

In [0]:
# Validate customer data from techmart_dw
import re
import pandas as pd

customers = spark.table("techmart_dw.customers").limit(1000).toPandas()

# Email validation
email_pattern = re.compile(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")
customers["email_valid"] = customers["email"].apply(
    lambda x: bool(email_pattern.match(str(x))) if pd.notna(x) else False
)

# Phone validation
phone_pattern = re.compile(r"^\+1-555-\d{4}$")
customers["phone_valid"] = customers["phone"].apply(
    lambda x: bool(phone_pattern.match(str(x))) if pd.notna(x) else False
)

print("Validation results:")
print(f"  Email valid: {customers['email_valid'].sum()}/{len(customers)} ({customers['email_valid'].mean()*100:.1f}%)")
print(f"  Phone valid: {customers['phone_valid'].sum()}/{len(customers)} ({customers['phone_valid'].mean()*100:.1f}%)")

In [0]:
# ============================================
# 🎯 YOUR TURN: Validate Multiple Formats
# ============================================
# Write regex patterns to validate:
# 1. US ZIP code: 5 digits or 5+4 format (12345 or 12345-6789)
# 2. ISO date: YYYY-MM-DD
# 3. IPv4 address: 0-255.0-255.0-255.0-255
#
# Test each with 3 valid and 2 invalid examples
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
import re

patterns = {
    "ZIP": re.compile(r"^\d{5}(-\d{4})?$"),
    "ISO Date": re.compile(r"^\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])$"),
    "IPv4": re.compile(r"^((25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}(25[0-5]|2[0-4]\d|[01]?\d\d?)$"),
}

tests = {
    "ZIP": ["12345", "12345-6789", "90210", "1234", "123456"],
    "ISO Date": ["2024-06-15", "2024-01-01", "2024-12-31", "2024-13-01", "24-06-15"],
    "IPv4": ["192.168.1.1", "10.0.0.1", "255.255.255.0", "256.1.1.1", "1.2.3"],
}

for name, pattern in patterns.items():
    print(f"\n{name}:")
    for test in tests[name]:
        result = "✅" if pattern.match(test) else "❌"
        print(f"  {result} '{test}'")

---
# 📗 Section 5: Log Parsing

In [0]:
# Parse Apache access log format
import re

apache_logs = [
    '192.168.1.1 - - [15/Jun/2024:10:30:01 +0000] "GET /api/orders HTTP/1.1" 200 1234',
    '10.0.0.5 - admin [15/Jun/2024:10:30:05 +0000] "POST /api/payments HTTP/1.1" 201 567',
    '172.16.0.1 - - [15/Jun/2024:10:30:10 +0000] "GET /api/products HTTP/1.1" 404 89',
]

apache_pattern = re.compile(
    r'(?P<ip>[\d.]+) - (?P<user>\S+) \[(?P<timestamp>[^\]]+)\] '
    r'"(?P<method>\w+) (?P<path>\S+) (?P<protocol>[^"]+)" '
    r'(?P<status>\d+) (?P<bytes>\d+)'
)

print("Parsed Apache logs:")
for log in apache_logs:
    match = apache_pattern.match(log)
    if match:
        d = match.groupdict()
        print(f"  {d['ip']:<15} {d['method']:<5} {d['path']:<20} {d['status']}")

---
# 📗 Section 6: PySpark Regex Functions

In [0]:
from pyspark.sql.functions import *

spark.sql("USE techmart_dw")
customers = spark.table("customers")

# regexp_extract: pull out email domain
extracted = (customers
    .filter("email IS NOT NULL")
    .withColumn("email_domain", regexp_extract("email", r"@(.+)$", 1))
    .withColumn("email_user", regexp_extract("email", r"^([^@]+)@", 1))
)
extracted.select("customer_id", "email", "email_domain", "email_user").show(5, truncate=False)

In [0]:
# regexp_replace: clean and standardize
cleaned = (customers
    .withColumn("name_clean", regexp_replace("first_name", r"[^a-zA-Z]", ""))
    .withColumn("phone_digits", regexp_replace("phone", r"[^0-9]", ""))
)
cleaned.select("customer_id", "first_name", "name_clean", "phone", "phone_digits").show(5, truncate=False)

In [0]:
# rlike: regex filtering
gmail_customers = customers.filter(col("email").rlike(r".*@gmail\.com$"))
print(f"Gmail customers: {gmail_customers.count()}")

# Find customers with ALL CAPS names (data quality issue)
caps_names = customers.filter(col("first_name").rlike(r"^[A-Z]+$"))
print(f"ALL CAPS names: {caps_names.count()}")

---
# 📗 Section 7: Common DE Regex Patterns (Cheat Sheet)

| Pattern | Regex |
|---|---|
| Email | `[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}` |
| Phone (US) | `\+?1?[-. ]?\(?\d{3}\)?[-. ]?\d{3}[-. ]?\d{4}` |
| IPv4 | `\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b` |
| ISO Date | `\d{4}-\d{2}-\d{2}` |
| URL | `https?://[^\s<>"']+` |
| SSN | `\b\d{3}-\d{2}-\d{4}\b` |
| Credit Card | `\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b` |
| UUID | `[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}` |

---
# 🚀 Section 8: Mini Projects

## Project 1: Log Parser Pipeline

In [0]:
# Parse app_logs from techmart_dw using regex
import re
from pyspark.sql.functions import *

logs = spark.table("techmart_dw.app_logs").limit(5000)

# Parse message field for structured extraction
parsed_logs = (logs
    .withColumn("has_error_code", col("message").rlike(r"\b[A-Z]{3}-\d{3}\b"))
    .withColumn("contains_ip", col("message").rlike(r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}"))
    .withColumn("message_category",
        when(col("message").rlike(r"(?i)timeout|connection"), "connectivity")
        .when(col("message").rlike(r"(?i)invalid|error"), "validation")
        .when(col("message").rlike(r"(?i)rate.?limit"), "throttling")
        .when(col("message").rlike(r"(?i)success|processed|passed"), "success")
        .otherwise("other")
    )
)

# Summary
parsed_logs.groupBy("message_category", "log_level").count().orderBy("count", ascending=False).show(10)
parsed_logs.write.format("delta").mode("overwrite").saveAsTable("techmart_dw.parsed_app_logs")
print(f"✅ Parsed {parsed_logs.count():,} log entries")

## Project 2: PII Detection & Masking

In [0]:
# PII detection and masking engine
from pyspark.sql.functions import *

customers = spark.table("techmart_dw.customers")

# Detect PII patterns
pii_detected = (customers
    .withColumn("has_email", col("email").isNotNull())
    .withColumn("has_phone", col("phone").isNotNull())
    # Mask email: show first 2 chars + domain
    .withColumn("email_masked",
        when(col("email").isNotNull(),
            concat(
                substring("email", 1, 2),
                lit("***@"),
                regexp_extract("email", r"@(.+)$", 1)
            )
        )
    )
    # Mask phone: show last 4 digits
    .withColumn("phone_masked",
        when(col("phone").isNotNull(),
            concat(lit("***-***-"), substring("phone", -4, 4))
        )
    )
)

print("PII Detection & Masking:")
pii_detected.select("customer_id", "email", "email_masked", "phone", "phone_masked").show(5, truncate=False)

# PII audit report
total = customers.count()
email_count = customers.filter("email IS NOT NULL").count()
phone_count = customers.filter("phone IS NOT NULL").count()
print(f"\nPII Audit Report:")
print(f"  Total records: {total:,}")
print(f"  Records with email: {email_count:,} ({email_count/total*100:.1f}%)")
print(f"  Records with phone: {phone_count:,} ({phone_count/total*100:.1f}%)")

---
# 🏆 Section 9: Interview Questions

## Q1: Regex to validate email?

```python
r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
```
Note: This covers 99% of cases. True email validation per RFC 5322 is extremely complex.

## Q2: Parse this log line?

`2024-06-15 10:30:01 ERROR auth-service Connection timeout after 30s`

```python
r"(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) (?P<level>\w+) (?P<service>[\w-]+) (?P<message>.+)"
```

## Q3: Regex performance on large datasets?

1. **re.compile()** — compile once, reuse (2-5x faster)
2. **Avoid .*** — use specific character classes
3. **Avoid nested quantifiers** — causes exponential backtracking
4. **In PySpark:** regex runs in parallel across partitions (scales linearly)
5. **Alternative:** Use string methods (startswith, split) when possible (faster)

## Q4: Regex vs dedicated parsers?

- **Use regex:** Simple patterns, validation, extraction from known formats
- **Use parsers:** HTML/XML (BeautifulSoup), JSON (json module), CSV (csv module)
- **Rule:** If the format has a standard parser, use it. Regex for everything else.

## Q5: Catastrophic backtracking?

Nested quantifiers like `(a+)+` or `(.*a){10}` cause exponential time on non-matching strings.
The regex engine tries every possible combination before failing.
**Fix:** Use specific character classes, avoid nested repetition, set timeouts.

## Q6: PII detection with regex?

Scan text columns with patterns for SSN, credit cards, emails, phones.
Flag records, mask values (partial replacement), generate audit report.
Run as a Silver-layer quality check before data reaches Gold.

## Q7: re.match() vs re.search()?

- `re.match()` — only matches at the START of the string
- `re.search()` — matches ANYWHERE in the string
- Use `match` for validation (entire string must match with `$`)
- Use `search` for extraction (find pattern within larger text)

## Q8: Python re vs PySpark Java regex?

Mostly compatible, but differences:
- Python: `\b` for word boundary. Java: same.
- Python: named groups `(?P<name>...)`. Java: `(?<name>...)` (no P).
- Python: `re.IGNORECASE` flag. PySpark: `(?i)` inline flag.
- PySpark `regexp_extract` uses Java regex (group indexing starts at 1).

---
# 📗 Section 5: Advanced Regex Patterns for DE

These patterns cover the most common data cleaning and extraction
tasks in production DE pipelines.

In [0]:
import re
from typing import Optional, Dict, List, Tuple

# ============================================================
# PATTERN LIBRARY: Common DE Regex Patterns
# ============================================================

PATTERNS = {
    # Identifiers
    "email": r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$",
    "phone_us": r"^\+?1?[\s\-\.]?\(?([0-9]{3})\)?[\s\-\.]?([0-9]{3})[\s\-\.]?([0-9]{4})$",
    "ssn": r"^(?!000|666|9\d{2})\d{3}-(?!00)\d{2}-(?!0000)\d{4}$",
    "credit_card": r"^(?:4[0-9]{12}(?:[0-9]{3})?|5[1-5][0-9]{14}|3[47][0-9]{13})$",
    "ip_v4": r"^(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)$",
    
    # Dates
    "date_iso": r"^\d{4}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01])$",
    "date_us": r"^(?:0[1-9]|1[0-2])/(?:0[1-9]|[12]\d|3[01])/\d{4}$",
    "datetime_iso": r"^\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}:\d{2}(?:\.\d+)?(?:Z|[+-]\d{2}:?\d{2})?$",
    
    # Financial
    "currency_usd": r"^\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?$",
    "percentage": r"^\d+(?:\.\d+)?%$",
    
    # Identifiers
    "uuid": r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$",
    "aws_arn": r"^arn:[a-z0-9\-]+:[a-z0-9\-]+:[a-z0-9\-]*:[0-9]{12}:[a-zA-Z0-9\-_/:.]+$",
    
    # Data quality
    "whitespace_only": r"^\s+$",
    "has_sql_injection": r"(?i)(\bSELECT\b|\bDROP\b|\bINSERT\b|\bDELETE\b|\bUNION\b|--|;)",
    "has_html": r"<[^>]+>",
}

def validate_field(value: str, pattern_name: str) -> bool:
    """Validate a value against a named pattern."""
    pattern = PATTERNS.get(pattern_name)
    if not pattern:
        raise ValueError(f"Unknown pattern: {pattern_name}")
    return bool(re.match(pattern, str(value), re.IGNORECASE))

# Test the pattern library
test_values = {
    "email": ["user@example.com", "invalid-email", "test@domain.co.uk"],
    "date_iso": ["2024-01-15", "2024-13-01", "not-a-date"],
    "ip_v4": ["192.168.1.1", "256.0.0.1", "10.0.0.1"],
    "uuid": ["550e8400-e29b-41d4-a716-446655440000", "not-a-uuid"],
}

for pattern_name, values in test_values.items():
    print(f"\n{pattern_name}:")
    for v in values:
        result = validate_field(v, pattern_name)
        print(f"  {'✅' if result else '❌'} {v}")

In [0]:
# --- PII Detection & Masking Engine ---
class PIIDetector:
    """
    Detect and mask PII in text data.
    Essential for GDPR/CCPA compliance in DE pipelines.
    """
    
    PII_PATTERNS = {
        "email": (
            r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
            lambda m: f"{m.group()[0]}***@***.{m.group().split('.')[-1]}"
        ),
        "phone": (
            r"\b(?:\+?1[\s\-\.]?)?\(?\d{3}\)?[\s\-\.]?\d{3}[\s\-\.]?\d{4}\b",
            lambda m: "***-***-" + m.group()[-4:]
        ),
        "ssn": (
            r"\b\d{3}-\d{2}-\d{4}\b",
            lambda m: "***-**-" + m.group()[-4:]
        ),
        "credit_card": (
            r"\b(?:\d{4}[\s\-]?){3}\d{4}\b",
            lambda m: "**** **** **** " + re.sub(r"[\s\-]", "", m.group())[-4:]
        ),
        "ip_address": (
            r"\b(?:\d{1,3}\.){3}\d{1,3}\b",
            lambda m: "***.***.***.***"
        ),
    }
    
    def detect(self, text: str) -> Dict[str, List[str]]:
        """Find all PII in text. Returns dict of {pii_type: [matches]}."""
        found = {}
        for pii_type, (pattern, _) in self.PII_PATTERNS.items():
            matches = re.findall(pattern, text)
            if matches:
                found[pii_type] = matches
        return found
    
    def mask(self, text: str) -> Tuple[str, Dict[str, int]]:
        """Mask all PII in text. Returns (masked_text, counts)."""
        result = text
        counts = {}
        for pii_type, (pattern, masker) in self.PII_PATTERNS.items():
            original = result
            result = re.sub(pattern, masker, result)
            count = len(re.findall(pattern, original))
            if count > 0:
                counts[pii_type] = count
        return result, counts

detector = PIIDetector()

# Test with sample text containing PII
sample_text = """
Customer John Doe (john.doe@example.com) called from 555-123-4567.
His SSN is 123-45-6789 and credit card 4532-1234-5678-9012.
IP address: 192.168.1.100. Secondary email: jane@company.org.
"""

print("Original text:", sample_text)
found = detector.detect(sample_text)
print("\nPII detected:")
for pii_type, matches in found.items():
    print(f"  {pii_type}: {matches}")

masked, counts = detector.mask(sample_text)
print("\nMasked text:", masked)
print("Masking counts:", counts)

In [0]:
# --- Log parsing with regex ---
import re
from datetime import datetime

# Apache Combined Log Format
APACHE_LOG_PATTERN = re.compile(
    r'(?P<ip>\S+) \S+ \S+ \[(?P<time>[^\]]+)\] '
    r'"(?P<method>\S+) (?P<path>\S+) \S+" '
    r'(?P<status>\d{3}) (?P<size>\d+|-) '
    r'"(?P<referer>[^"]*)" "(?P<user_agent>[^"]*)"'
)

def parse_apache_log(line: str) -> Optional[dict]:
    """Parse a single Apache log line."""
    match = APACHE_LOG_PATTERN.match(line)
    if not match:
        return None
    
    d = match.groupdict()
    return {
        "ip": d["ip"],
        "timestamp": datetime.strptime(d["time"], "%d/%b/%Y:%H:%M:%S %z"),
        "method": d["method"],
        "path": d["path"],
        "status_code": int(d["status"]),
        "response_size": int(d["size"]) if d["size"] != "-" else 0,
        "referer": d["referer"] if d["referer"] != "-" else None,
        "user_agent": d["user_agent"],
        "is_error": int(d["status"]) >= 400,
        "is_bot": bool(re.search(r"bot|crawler|spider", d["user_agent"], re.I)),
    }

# Test log lines
log_lines = [
    '192.168.1.1 - - [15/Jan/2024:10:30:00 +0000] "GET /api/orders HTTP/1.1" 200 1234 "-" "Mozilla/5.0"',
    '10.0.0.5 - - [15/Jan/2024:10:31:00 +0000] "POST /api/checkout HTTP/1.1" 201 567 "https://shop.com" "Chrome/120"',
    '192.168.1.2 - - [15/Jan/2024:10:32:00 +0000] "GET /admin HTTP/1.1" 403 89 "-" "Googlebot/2.1"',
    'bad log line that won\'t parse',
]

parsed = [parse_apache_log(line) for line in log_lines]
for i, (line, result) in enumerate(zip(log_lines, parsed)):
    if result:
        print(f"Line {i+1}: {result['method']} {result['path']} → {result['status_code']} (bot={result['is_bot']})")
    else:
        print(f"Line {i+1}: ❌ Failed to parse")

---
# 📗 Section 6: Practice Exercises

In [0]:
# ============================================================
# EXERCISE 1: Data Standardization Pipeline
# ============================================================
# Clean and standardize a messy customer dataset using regex

import pandas as pd

messy_customers = pd.DataFrame({
    "customer_id": range(1, 9),
    "phone": ["(555) 123-4567", "555.987.6543", "+1-800-555-0100",
              "5551234567", "555 456 7890", "not-a-phone", "555-123", "1(800)5550199"],
    "email": ["User@Example.COM", "  test@domain.org  ", "ADMIN@COMPANY.NET",
              "invalid-email", "user+tag@sub.domain.co", "another@test.com",
              "no-at-sign", "valid@email.io"],
    "postal_code": ["98101", "98101-1234", "SW1A 1AA", "10001", "not-postal",
                    "90210", "V6B 4N6", "12345-6789"],
})

def standardize_phone(phone: str) -> Optional[str]:
    """Standardize to E.164 format: +1XXXXXXXXXX"""
    if not phone:
        return None
    digits = re.sub(r"\D", "", phone)
    if digits.startswith("1") and len(digits) == 11:
        digits = digits[1:]
    if len(digits) == 10:
        return f"+1{digits}"
    return None  # Invalid

def standardize_email(email: str) -> Optional[str]:
    """Lowercase, strip, validate."""
    if not email:
        return None
    cleaned = email.strip().lower()
    if re.match(r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$", cleaned):
        return cleaned
    return None

def extract_postal_digits(postal: str) -> Optional[str]:
    """Extract 5-digit US ZIP or return None."""
    match = re.search(r"\b(\d{5})(?:-\d{4})?\b", postal)
    return match.group(1) if match else None

messy_customers["phone_clean"] = messy_customers["phone"].apply(standardize_phone)
messy_customers["email_clean"] = messy_customers["email"].apply(standardize_email)
messy_customers["zip_code"] = messy_customers["postal_code"].apply(extract_postal_digits)

print("Standardized customer data:")
cols = ["customer_id", "phone", "phone_clean", "email", "email_clean", "postal_code", "zip_code"]
print(messy_customers[cols].to_string(index=False))

print(f"\nPhone success rate: {messy_customers['phone_clean'].notna().sum()}/{len(messy_customers)}")
print(f"Email success rate: {messy_customers['email_clean'].notna().sum()}/{len(messy_customers)}")

---
# 📗 Section 5: Regex in PySpark

## Using Regex in Spark SQL and PySpark

```python
from pyspark.sql.functions import regexp_extract, regexp_replace, rlike, col

# Extract email domain
df.withColumn("domain", regexp_extract(col("email"), r"@(.+)$", 1))

# Validate email format
df.filter(col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"))

# Clean phone numbers (remove non-digits)
df.withColumn("phone_clean", regexp_replace(col("phone"), r"[^0-9]", ""))

# Extract year from date string
df.withColumn("year", regexp_extract(col("date_str"), r"(\d{4})", 1))
```

## Common DE Regex Patterns

| Pattern | Regex | Use Case |
|---------|-------|---------|
| Email | `^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$` | Validate emails |
| Phone (US) | `^\+?1?[-.\s]?\(?[0-9]{3}\)?[-.\s]?[0-9]{3}[-.\s]?[0-9]{4}$` | Validate phones |
| IP Address | `^(?:[0-9]{1,3}\.){3}[0-9]{1,3}$` | Parse logs |
| Date (ISO) | `^\d{4}-\d{2}-\d{2}$` | Validate dates |
| UUID | `^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$` | Validate IDs |
| Credit Card | `^\d{4}[- ]?\d{4}[- ]?\d{4}[- ]?\d{4}$` | Detect PII |
| SSN | `^\d{3}-\d{2}-\d{4}$` | Detect PII |

In [0]:
# Regex patterns for data engineering
import re

print("Regex Patterns for Data Engineering")
print("=" * 60)

# Common patterns
patterns = {
    "email": r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
    "phone_us": r"^\+?1?[-.\s]?\(?[0-9]{3}\)?[-.\s]?[0-9]{3}[-.\s]?[0-9]{4}$",
    "date_iso": r"^\d{4}-\d{2}-\d{2}$",
    "ip_address": r"^(?:[0-9]{1,3}\.){3}[0-9]{1,3}$",
    "uuid": r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$",
    "credit_card": r"^\d{4}[- ]?\d{4}[- ]?\d{4}[- ]?\d{4}$",
}

test_values = {
    "email": ["alice@example.com", "invalid-email", "user@domain.co.uk"],
    "phone_us": ["555-867-5309", "(555) 867-5309", "5558675309", "not-a-phone"],
    "date_iso": ["2024-03-15", "15/03/2024", "March 15 2024"],
    "ip_address": ["192.168.1.1", "256.0.0.1", "10.0.0.1"],
}

for pattern_name, test_vals in test_values.items():
    pattern = patterns[pattern_name]
    print(f"\n{pattern_name}:")
    for val in test_vals:
        match = bool(re.match(pattern, val))
        icon = "✅" if match else "❌"
        print(f"  {icon} '{val}'")

# Data cleaning with regex
print("\n\nData Cleaning Examples:")
dirty_data = [
    "  Alice Smith  ",
    "JOHN DOE",
    "bob.jones@EXAMPLE.COM",
    "555-867-5309",
    "$1,234.56",
]

print("  Original -> Cleaned:")
for val in dirty_data:
    cleaned = val.strip()
    cleaned = re.sub(r"\s+", " ", cleaned)
    cleaned = cleaned.title() if "@" not in cleaned else cleaned.lower()
    cleaned = re.sub(r"[^0-9]", "", cleaned) if re.match(r"[\d\-\(\)\s]+", cleaned) else cleaned
    print(f"  '{val}' -> '{cleaned}'")


---
# ✅ Notebook Complete!

**What was covered:**
- Regex fundamentals: character classes, quantifiers, groups, anchors
- Python re module: search, findall, sub, compile, named groups
- Data validation: email, phone, ZIP, date, IP patterns
- Log parsing: Apache format, application logs with named groups
- PySpark regex: regexp_extract, regexp_replace, rlike
- Cheat sheet of common DE patterns
- 2 mini projects: Log Parser Pipeline, PII Detection & Masking
- 8 interview questions

**Next:** Notebook 17 — SQL Connectors for Data Engineering

---